# Medical Image Denoising - train on a free GPU
Runtime -> Change runtime type -> **T4 GPU**, then run all cells.

In [ ]:
!git clone https://github.com/srivastavashreyaa/Medical-Image-Denoising-UNet.git
%cd Medical-Image-Denoising-UNet
!pip install -q -r requirements.txt kagglehub

## 1. Download a dataset
COVID-19 Radiography Database (public, Kaggle). You may need to accept the dataset terms on Kaggle and set `KAGGLE_USERNAME` / `KAGGLE_KEY` first.

In [ ]:
import kagglehub, glob, os, shutil
path = kagglehub.dataset_download('tawsifurrahman/covid19-radiography-database')
print(path)
os.makedirs('raw_images', exist_ok=True)
files = glob.glob(path + '/**/images/*.png', recursive=True)[:3000]   # subset keeps training fast
for i, f in enumerate(files):
    shutil.copy(f, f'raw_images/img_{i:05d}.png')
print(len(files), 'images copied')

## 2. Preprocess -> Clean/Noisy pairs

In [ ]:
!python preprocess.py --input_dir raw_images --output_dir data --img_size 256 --augment 1 --noise_type mixed

## 3. Train + evaluate

In [ ]:
!python train.py --data_dir data --epochs 30 --batch_size 16

## 4. Benchmark, plots, qualitative figure

In [ ]:
!python benchmark.py --data_dir data
!python visualize.py
!python compare.py --n 4

In [ ]:
from IPython.display import Image, display
display(Image('results/qualitative_comparison.png'))

## 5. Download results
Commit `denoising_metrics.csv`, `training_history.png`, `results/` and the model to your repo.

In [ ]:
!zip -rq outputs.zip denoising_model.keras denoising_metrics.csv training_history.png results
from google.colab import files
files.download('outputs.zip')